# Step 4 · Screen sampled pathogen–host evidence

Screen a reproducible sample of PubMed records for each configured pathogen–host pair. A pair is accepted as soon as one completed sample assessment answers yes with a supporting span quoted exactly from that article’s abstract. A title alone cannot establish acceptance. Records without abstracts remain unassessed and keep the pair incomplete unless another abstract supports it. The sample size is set in `00_configure_run` and cannot exceed 200 records per pair. Full mode prepares the sample for every configured pair with PubMed hits, but retrieves abstracts only for those sampled records. The handler is provider-agnostic; model screening runs only when enabled.

## Step 1 · Load the selected run, pair cache, and sampling policy

In [1]:
import json, os, sys
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, ncbi_contact_email
from pathogen_atlas_v4.runs import resolve_run
from pathogen_atlas_v4.pubmed_client import prepare_pair_screen_samples, ncbi_api_key, literature_cache_key
from pathogen_atlas_v4.pathogen_screening import build_pathogen_screen_tasks, select_pathogen_host_pairs
from pathogen_atlas_v4.decision_runner import run_decisions, decision_cache_key_for_task
from pathogen_atlas_v4.jev_pathogen_screen import make_jev_pathogen_screen_handler, openrouter_key_available
CONFIG = yaml.safe_load((PROJECT_ROOT / 'configs/pilot_v1.yaml').read_text(encoding='utf-8'))
RUN_ID = configured_run_id(PROJECT_ROOT)
RUN_MODE = str(CONFIG['run'].get('selection_mode', '')).casefold()
IS_FULL_RUN = RUN_MODE == 'full'
ROOT, LOG, LOG_PATH = configure_notebook('04_screen_pathogen_host_pairs', RUN_ID)
STAGING_TAXONOMY_ROOT = sorted((ROOT / 'data/v4/staging').glob('*/taxonomy'))[-1]
RUN_DIR = ROOT / 'data/v4/runs' / RUN_ID
RUN_TAXONOMY_ROOT = RUN_DIR / 'taxonomy'
if not (RUN_TAXONOMY_ROOT / 'pathogens.parquet').exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
PATHOGENS = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet')
HOSTS = pd.read_parquet(STAGING_TAXONOMY_ROOT / 'hosts.parquet')
ALIASES = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogen_aliases.parquet', columns=['pathogen_id','name'])
host_groups = {k: g.host_id.astype(str).tolist() for k,g in HOSTS.groupby('host_group')}
pathogen_groups = {k: g.pathogen_id.astype(str).tolist() for k,g in PATHOGENS.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(HOSTS.host_id.astype(str)), known_pathogen_ids=set(PATHOGENS.pathogen_id.astype(str)), host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
SAMPLING = CONFIG['acquisition'].get('sampling', {})
SAMPLE_MAXIMUM = SAMPLING.get('max_pmids_per_pair', 200)
if not isinstance(SAMPLE_MAXIMUM, int) or not 1 <= SAMPLE_MAXIMUM <= 200:
    raise ValueError('Pathogen screening is capped at 200 abstracts per pair; set max_pmids_per_pair to an integer from 1 to 200 in 00_configure_run.')
SAMPLE_OLDEST = int(SAMPLING.get('oldest_pmids_per_pair', 30))
SAMPLE_SEED = int(SAMPLING.get('random_seed', 42))
SAMPLE_POLICY = str(SAMPLING.get('sample_policy_version', 'stable-prefix-hash-v1'))
cache_key = literature_cache_key(query_version=CONFIG['acquisition']['query_strategy_version'],
                                  pathogen_literature_query_version=CONFIG['acquisition']['pathogen_literature_query_version'],
                                  taxonomy_snapshot=STAGING_TAXONOMY_ROOT.parent.name)
CACHE_DIR = ROOT / 'data/v4/cache/literature' / cache_key
PROMPTS = yaml.safe_load((ROOT / CONFIG['decisions']['prompt_catalogue']).read_text(encoding='utf-8'))
PAIR_SEARCHES_PATH = CACHE_DIR / 'tables/pair_searches.parquet'
SAMPLE_MATCHES_PATH = CACHE_DIR / 'tables/pathogen_screen_matches.parquet'
ARTICLES_PATH = CACHE_DIR / 'tables/articles.parquet'
SHARED_DECISION_CACHE = ROOT / 'data/v4/cache/decisions'
SCREEN_LEDGER_PATH = SHARED_DECISION_CACHE / 'pathogen_screen_ledger.parquet'
PAIR_SELECTION_PATH = RUN_DIR / 'pathogen_host_selection.parquet'
print('Run:', RUN_ID, '| candidate pairs:', len(resolved.pairs), '| sample cap:', SAMPLE_MAXIMUM, '| oldest PMID IDs:', SAMPLE_OLDEST, '| seed:', SAMPLE_SEED)
print('Pair cache:', CACHE_DIR, '| log:', LOG_PATH)
LOG.info('Pathogen screen setup: pairs=%d cap=%s oldest=%d seed=%d cache=%s', len(resolved.pairs), SAMPLE_MAXIMUM, SAMPLE_OLDEST, SAMPLE_SEED, CACHE_DIR)

2026-11-02 17:37 INFO Notebook session started at 2026-11-02T17:37:08.600838+00:00
2026-11-02 17:37 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-02 17:37 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02b_pathogen_screening.log
2026-11-02 17:37 INFO Pathogen screen setup: pairs=17554 cap=200 oldest=30 seed=42 cache=/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/bb96506e44b71651c082c7271ad7b38cc6cc96ec75300e0ab956ddc34a3da29c


Run: pilot-v1 | candidate pairs: 17554 | sample cap: 200 | oldest PMID IDs: 30 | seed: 42
Pair cache: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/bb96506e44b71651c082c7271ad7b38cc6cc96ec75300e0ab956ddc34a3da29c | log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02b_pathogen_screening.log


## Step 2 · Select stable sample PMIDs and retrieve their abstracts

In [2]:
RUN_SAMPLE_PREPARATION = IS_FULL_RUN  # Full mode always prepares every selected pair; the operation resumes from cache.
if RUN_SAMPLE_PREPARATION:
    NCBI_EMAIL = ncbi_contact_email(ROOT)
    NCBI_API_KEY = ncbi_api_key()
    sample_summary = prepare_pair_screen_samples(
        resolved.pairs, CACHE_DIR, sample_maximum=SAMPLE_MAXIMUM, oldest_count=SAMPLE_OLDEST,
        seed=SAMPLE_SEED, sample_policy_version=SAMPLE_POLICY, email=NCBI_EMAIL,
        api_key=NCBI_API_KEY, logger=LOG)
    display(sample_summary)
else:
    print('Sample preparation skipped. Set RUN_SAMPLE_PREPARATION=True to prepare the pair samples.')
    print('Existing sample table:', SAMPLE_MATCHES_PATH.exists())
if SAMPLE_MATCHES_PATH.exists():
    sample_matches = pd.read_parquet(SAMPLE_MATCHES_PATH)
    articles = pd.read_parquet(ARTICLES_PATH)
    selected_pair_keys = set((str(pathogen_id), str(host_id)) for pathogen_id, host_id in resolved.pairs)
    if not sample_matches.empty:
        sample_pair_index = pd.MultiIndex.from_frame(sample_matches[['pathogen_id', 'host_id']].astype(str))
        sample_matches = sample_matches.loc[sample_pair_index.isin(selected_pair_keys)].copy()
    print('Selected-run sample records:', f'{len(sample_matches):,}', '| abstracts available:', f'{int(sample_matches.abstract_available.sum()):,}')
else:
    sample_matches = pd.DataFrame(columns=['pathogen_id','host_id','pmid','query_hash','query_version','pair_search_count','sample_selection'])
    articles = pd.DataFrame(columns=['pmid','title','abstract'])
if IS_FULL_RUN:
    if not PAIR_SEARCHES_PATH.exists():
        raise FileNotFoundError(f'Run Step 3 pair searches first: {PAIR_SEARCHES_PATH}')
    pair_searches = pd.read_parquet(PAIR_SEARCHES_PATH)
    pair_search_index = pd.MultiIndex.from_frame(pair_searches[['pathogen_id', 'host_id']].astype(str))
    pair_searches = pair_searches.loc[pair_search_index.isin(selected_pair_keys)].copy()
    acquired_pair_keys = set(map(tuple, pair_searches.loc[pair_searches.status.astype(str).eq('completed'), ['pathogen_id', 'host_id']].astype(str).drop_duplicates().to_numpy()))
    missing_pair_searches = selected_pair_keys - acquired_pair_keys
    if missing_pair_searches:
        raise RuntimeError(f'Full-run pair acquisition is incomplete: {len(missing_pair_searches):,} configured pairs lack a completed PubMed search. Rerun 03_acquire_pair_pubmed_records.')
    search_pair_keys = set(map(tuple, pair_searches.loc[pd.to_numeric(pair_searches.search_count, errors='coerce').fillna(0).gt(0), ['pathogen_id', 'host_id']].astype(str).to_numpy()))
    expected_sample_pairs = selected_pair_keys & search_pair_keys
    prepared_sample_pairs = set(map(tuple, sample_matches[['pathogen_id', 'host_id']].astype(str).drop_duplicates().to_numpy())) if not sample_matches.empty else set()
    missing_sample_pairs = expected_sample_pairs - prepared_sample_pairs
    if missing_sample_pairs:
        raise RuntimeError(f'Full run sample cache is incomplete: {len(missing_sample_pairs)} of {len(expected_sample_pairs)} PubMed-hit pairs have no sample. Rerun sample preparation.')
    print(f'Full-run sample coverage: {len(prepared_sample_pairs):,}/{len(expected_sample_pairs):,} PubMed-hit pairs.')
if not sample_matches.empty:
    sample_sizes = sample_matches.groupby(['pathogen_id', 'host_id']).pmid.nunique()
    oversized = sample_sizes[sample_sizes.gt(SAMPLE_MAXIMUM)]
    if not oversized.empty:
        raise RuntimeError(f'Sample cache exceeds the configured cap for {len(oversized)} pairs (cap={SAMPLE_MAXIMUM}); rerun sample preparation.')
LOG.info('Sample inputs loaded: rows=%d articles=%d', len(sample_matches), len(articles))

Using the configured Git email as the NCBI contact address.


2026-11-02 17:37 INFO Prepared pathogen screen samples: pairs=17554 sample_rows=59945 new_abstract_fetches=0 cap=200


{'pairs': 17554,
 'sample_rows': 59945,
 'new_abstract_fetches': 0,
 'sample_cap_per_pair': 200,
 'oldest_per_pair': 30,
 'sample_seed': 42,
 'sample_policy_version': 'stable-prefix-hash-v1',
 'outputs': '/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/cache/literature/bb96506e44b71651c082c7271ad7b38cc6cc96ec75300e0ab956ddc34a3da29c/tables'}

2026-11-02 17:37 INFO Sample inputs loaded: rows=59945 articles=25223


Selected-run sample records: 59,945 | abstracts available: 58,915
Full-run sample coverage: 1,643/1,643 PubMed-hit pairs.


## Step 3 · Build and inspect pathogen yes/no tasks

In [3]:
PROMPT_MODEL_VERSION = str(CONFIG['decisions'].get('model_version', 'configured-per-run'))
TASKS = build_pathogen_screen_tasks(sample_matches, articles, PATHOGENS, HOSTS, PROMPTS,
    run_id=RUN_ID, run_config_version=str(CONFIG['run']['config_version']), model_version=PROMPT_MODEL_VERSION)
print('Screen tasks:', f'{len(TASKS):,}')
if TASKS:
    print(json.dumps(TASKS[0]['prompt'], indent=2, ensure_ascii=False))
LOG.info('Built pathogen screen tasks=%d', len(TASKS))

2026-11-02 17:37 INFO Built pathogen screen tasks=58915


Screen tasks: 58,915
{
  "question_version": "pathogen-screen-v2",
  "prompt_version": "pathogen-screen-prompt-v2",
  "system": "Assess only the supplied PubMed title and abstract for the specified organism and selected host. Answer yes only when this record directly supports that the organism causes disease in that host. A positive answer requires an exact supporting passage from the abstract; the title alone is not sufficient. A mere mention, colonization or carriage without disease, vector or reservoir role, experimental use, or food contamination alone is not sufficient. Do not infer beyond the supplied text. Return valid JSON matching the requested schema.",
  "user": "Question: Does this PubMed record provide direct evidence that the named organism causes disease in the selected host?\n\nNamed organism: Potyvirus achyranthis\nPathogen ID: viruses:00136e62d974a68c1a27\nSelected host: Oryza sativa\nHost ID: ncbi:4530\n\nArticle title:\nCombined Control of Rice Sheath Blight Using a

## Step 4 · Run or resume the screening ledger

In [ ]:
SCREENING_MODEL = str(CONFIG.get('decisions', {}).get('model_version', 'typesafe/jev-1.14'))
SCREENING_ENABLED = bool(CONFIG.get('decisions', {}).get('screen_pathogen_pairs', IS_FULL_RUN))
TASK_CACHE_KEYS = {decision_cache_key_for_task(task) for task in TASKS}
EXISTING_LEDGER = pd.read_parquet(SCREEN_LEDGER_PATH) if SCREEN_LEDGER_PATH.exists() else pd.DataFrame()
if not EXISTING_LEDGER.empty and {'cache_key', 'status'}.issubset(EXISTING_LEDGER.columns):
    completed_cache_keys = set(EXISTING_LEDGER.loc[
        EXISTING_LEDGER.status.astype(str).eq('completed'), 'cache_key'
    ].astype(str))
else:
    completed_cache_keys = set()
PENDING_TASKS = [task for task in TASKS if decision_cache_key_for_task(task) not in completed_cache_keys]
PATHOGEN_SCREEN_HANDLER = None
MAX_ATTEMPTS = int(CONFIG.get('logging', {}).get('max_attempts', CONFIG.get('decisions', {}).get('max_attempts', 2)))
MAX_SCREEN_WORKERS = int(CONFIG.get('logging', {}).get('decision_workers', 1024))
SCREEN_CHECKPOINT_EVERY = int(CONFIG.get('logging', {}).get('decision_checkpoint_every', 1000))
if SCREENING_ENABLED and PENDING_TASKS:
    if not openrouter_key_available():
        raise RuntimeError('Screening has pending tasks but OPENROUTER_API_KEY is not set and no OpenRouter Keychain key was found.')
    PATHOGEN_SCREEN_HANDLER = make_jev_pathogen_screen_handler(model=SCREENING_MODEL)
    print(f'Screening {len(PENDING_TASKS):,} remaining abstracts with {SCREENING_MODEL}; workers={MAX_SCREEN_WORKERS}.')
    LEDGER = run_decisions(PENDING_TASKS, PATHOGEN_SCREEN_HANDLER, ledger_path=SCREEN_LEDGER_PATH,
                           raw_response_path=SHARED_DECISION_CACHE / 'pathogen_screen_raw_responses.jsonl',
                           max_attempts=MAX_ATTEMPTS, max_workers=MAX_SCREEN_WORKERS,
                           checkpoint_every=SCREEN_CHECKPOINT_EVERY)
else:
    LEDGER = EXISTING_LEDGER
    if not SCREENING_ENABLED:
        print('Screening disabled by run configuration; set decisions.screen_pathogen_pairs=True in 00_configure_run for a full screen.')
    elif not TASKS:
        print('No sampled abstracts require model screening.')
    else:
        print('No provider calls needed; all sampled abstract decisions are already complete.')
if not LEDGER.empty and 'cache_key' in LEDGER:
    LEDGER = LEDGER.loc[LEDGER.cache_key.astype(str).isin(TASK_CACHE_KEYS)].copy()
else:
    LEDGER = pd.DataFrame()
if IS_FULL_RUN:
    completed_keys = set(LEDGER.loc[LEDGER.status.astype(str).eq('completed'), 'cache_key'].astype(str)) if not LEDGER.empty else set()
    missing_screenings = TASK_CACHE_KEYS - completed_keys
    if missing_screenings:
        raise RuntimeError(f'Full run screening is incomplete: {len(missing_screenings):,} available abstract tasks are not completed. Check decisions.screen_pathogen_pairs in configs/pilot_v1.yaml and OPENROUTER_API_KEY, then rerun this cell to resume from the ledger.')
    print(f'Full-run screening coverage: {len(completed_keys):,}/{len(TASK_CACHE_KEYS):,} available abstract tasks.')
print('Ledger:', SCREEN_LEDGER_PATH, '| rows:', len(LEDGER))
LOG.info('Pathogen screen ledger rows=%d pending_before_run=%d', len(LEDGER), len(PENDING_TASKS))

## Step 5 · Apply the acceptance rule for each host

In [5]:
if not sample_matches.empty:
    pair_selection = select_pathogen_host_pairs(resolved.pairs, sample_matches, LEDGER)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    pair_selection.to_parquet(PAIR_SELECTION_PATH, index=False)
    display(pair_selection.selection_status.value_counts(dropna=False).rename('pairs').to_frame())
    display(pair_selection.loc[pair_selection.selection_status.eq('accepted')].head(20))
    print('Accepted pathogen–host pairs:', int(pair_selection.selected_as_pathogen_for_host.sum()))
else:
    pair_selection = pd.DataFrame()
    pair_selection = select_pathogen_host_pairs(resolved.pairs, sample_matches, LEDGER)
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    pair_selection.to_parquet(PAIR_SELECTION_PATH, index=False)
    print('No abstract sample is available yet; candidate pairs are recorded as incomplete or without sampled records.')
LOG.info('Pair selection saved=%s accepted=%d', not pair_selection.empty, int(pair_selection.selected_as_pathogen_for_host.sum()) if not pair_selection.empty else 0)

,pairs
selection_status,
no_sampled_records,15911
rejected,1349
incomplete,285
accepted,9


,pathogen_id,host_id,relationship_context,n_sampled_records,n_sampled_records_with_abstract,n_completed_screenings,n_pair_search_records,n_positive_records,positive_pmids,positive_evidence,selected_as_pathogen_for_host,selection_status
1976,viruses:1c7dcecfc400473012e4,ncbi:4530,disease,200,200,200,310,1,[36423156],"[{'pmid': '36423156', 'evidence_span': 'Rice s...",True,accepted
7297,viruses:68a4bbc3f3a5719b609d,ncbi:4530,disease,15,15,15,15,1,[17073305],"[{'pmid': '17073305', 'evidence_span': '(tall ...",True,accepted
7866,viruses:71279d492d6f0d6c2d14,ncbi:4530,disease,200,197,197,368,2,"[10665229, 32542993]","[{'pmid': '10665229', 'evidence_span': 'The ex...",True,accepted
9629,viruses:8b8976efac7e9c33ff11,ncbi:4530,disease,2,2,2,2,1,[36840129],"[{'pmid': '36840129', 'evidence_span': 'Exogen...",True,accepted
10121,viruses:928d0c303dc8deaf1b99,ncbi:4530,disease,45,45,45,45,1,[26905382],"[{'pmid': '26905382', 'evidence_span': 'Sexual...",True,accepted
10469,viruses:976f27c4e726cd4225b8,ncbi:4530,disease,9,9,9,9,1,[41649689],"[{'pmid': '41649689', 'evidence_span': 'Here, ...",True,accepted
14408,viruses:d140a518d6637bf24c48,ncbi:4530,disease,182,180,180,182,2,"[23132204, 26296721]","[{'pmid': '26296721', 'evidence_span': 'Rice g...",True,accepted
16967,viruses:f77114fb0e9d224e3584,ncbi:4530,disease,200,186,186,478,7,"[10766310, 10948363, 2045788, 23763585, 335484...","[{'pmid': '10948363', 'evidence_span': 'The fl...",True,accepted
17185,viruses:fa9873798bea5617f432,ncbi:4530,disease,125,123,123,125,64,"[10404165, 11080631, 12679871, 14564505, 14730...","[{'pmid': '18700826', 'evidence_span': 'The At...",True,accepted


2026-11-02 17:37 INFO Pair selection saved=True accepted=9


Accepted pathogen–host pairs: 9


Acceptance is pair-specific: one completed yes response with an exact supporting span accepts that pathogen for that host. A pair with no yes is rejected only after every sampled PMID has a completed screen; otherwise it remains incomplete. No confidence threshold is applied to the yes/no acceptance rule.